## Summary

1. 229 features with likely to have high collinearity.
2. Coefficient of Correlations of each features to dependent variable does not shows significant clue.
3. Feature importance gives insight on features importance that helps on prediction.

#### Implementation for Basic Transformation:
1. Consider to drop column with high collinearity.
2. Consider to reduce features for prediction, objective to use less sensor on testing which achieving the similar performance.

In [3]:
import pandas as pd
import numpy as np
import missingno as msno
from pathlib import Path

curr_dir = Path.cwd()
proj_root = curr_dir.parent
raw_data = proj_root / "data/raw/uci-secom.csv"

df = pd.read_csv(raw_data)

In [13]:
# Correlation to Label
from scipy.stats import pointbiserialr

cols_to_exclude = ["Time"]
features_df = df.drop(columns=cols_to_exclude)
target = df['Pass/Fail']

results = {}

for col in features_df.columns:
    # pointbiserialr returns a tuple: (correlation_coefficient, p_value)
    corr, p_val = pointbiserialr(target, features_df[col])
    results[col] = {'Correlation': corr, 'P-Value': p_val, 'abs_corr': abs(corr)}
    
pbs_df = pd.DataFrame.from_dict(results, orient='index')
pbs_df = pbs_df.sort_values(by='abs_corr', ascending=False)

print("Correlation of features to the label:")
print(pbs_df[['Correlation','P-Value']][:30])

Correlation of features to the label:
           Correlation   P-Value
Pass/Fail     1.000000  0.000000
114           0.068655  0.006553
249           0.066478  0.008480
387           0.066315  0.008643
575          -0.052731  0.036872
573          -0.051873  0.040057
577          -0.049633  0.049487
115          -0.043654  0.084081
360           0.038608  0.126599
521           0.036722  0.146225
494           0.035182  0.163921
574          -0.034713  0.169608
359           0.033077  0.190645
572          -0.032233  0.202209
222           0.031294  0.215674
87           -0.030422  0.228754
576          -0.028488  0.259733
88            0.026865  0.287878
86            0.024974  0.323158
20            0.023253  0.357644
221           0.021609  0.392655
526          -0.021536  0.394248
254          -0.021509  0.394850
392          -0.021268  0.400164
120          -0.020277  0.422487
388           0.019723  0.435287
493           0.019420  0.442360
571          -0.019353  0.443934
117  

In [7]:
# Multi-Collinearity

# Calculate the correlation matrix
corr_matrix = features_df.corr(method='spearman').abs()

# Select upper triangle of correlation matrix (to avoid duplicates)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.85
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

print(f"There are {len(to_drop)} features that are highly redundant.")

There are 229 features that are highly redundant.
